# Mô hình: Basic CNN 1D
**Dataset**: Diabetes
**Bài tập**: Thực nghiệm CNN 1D trên dữ liệu Tabular


## 2. Cơ sở lý thuyết: Basic CNN 1D cho dữ liệu bảng
Việc sử dụng mạng tích chập (CNN) trên dữ liệu bảng (tabular data) có thể xem là một bài toán thực nghiệm. Thông thường, CNN dùng cho ảnh vì có tính không gian địa phương (spatial locality). Đối với dữ liệu bảng, ta coi 21 đặc trưng như một chuỗi 1D chiều dài 21.

Mạng Basic CNN 1D sử dụng tích chập 1 chiều (Conv1d).
Công thức cho phép tích chập 1D:
$$ y[i] = \sum_{k=0}^{K-1} w[k] x[i - k] + b $$
Cấu trúc mạng:
$$ \hat{y} = FC(Pool(ReLU(BN(Conv1d_2(Pool(ReLU(BN(Conv1d_1(X))))))))) $$


In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
EPOCHS = 20
BATCH_SIZE = 256
LR = 1e-3
IN_CHANNELS = 1
NUM_CLASSES = 1
DATASET_NAME = 'Diabetes'
MODEL_NAME = 'Basic CNN 1D'
MODEL_KEY = 'basic'


## 3. Data Loading & Preprocessing


In [ ]:
df = pd.read_csv('../intel-sys-assignment-04/dataset/diabets.csv')
# Lấy mẫu 50000 dòng để tính toán nhanh hơn
df = df.sample(n=50000, random_state=42)

X = df.drop(columns=['Diabetes_binary']).values
y = df['Diabetes_binary'].values

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Chuyển thành tensor và thêm chiều channel (batch, channel, features) -> (batch, 1, 21)
X_train_tensor = torch.FloatTensor(X_train).unsqueeze(1)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 4. Data Exploration


In [ ]:
print("Head of dataset:")
display(df.head())
print("\nDataset description:")
display(df.describe())


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Diabetes_binary')
plt.title("Class Distribution")
plt.show()


In [ ]:
plt.figure(figsize=(15, 12))
sns.heatmap(df.corr(), annot=False, cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()


## 5. Model Architecture


In [ ]:
class BasicCNN1D(nn.Module):
    """
    M1: Basic 1D CNN for tabular data
    ŷ = FC(Pool(ReLU(BN(Conv1d₂(Pool(ReLU(BN(Conv1d₁(X)))))))))
    """
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = BasicCNN1D(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)
print(model)


## 6. Training Functions


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze(-1)  # (batch, 1) -> (batch,)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        predicted = (torch.sigmoid(outputs) >= 0.5).float()
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs).squeeze(-1)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item() * inputs.size(0)
        probs = torch.sigmoid(outputs)
        predicted = (probs >= 0.5).float()
        
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        
    return total_loss / total, correct / total, all_preds, all_labels, all_probs


## 7. Training Loop


In [ ]:
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, _, _, _ = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")


## 8. Visualization


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train')
plt.plot(test_losses, label='Test')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train')
plt.plot(test_accs, label='Test')
plt.title('Accuracy over Epochs')
plt.legend()
plt.show()


## 9. Final Evaluation


In [ ]:
test_loss, test_acc, all_preds, all_labels, all_probs = evaluate(model, test_loader, criterion, device)

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['No Diabetes', 'Diabetes']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Diabetes', 'Diabetes'], yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.4f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()


## 10. Save Results


In [ ]:
os.makedirs('results', exist_ok=True)
results = {
    'model_name': MODEL_NAME,
    'dataset': DATASET_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LR,
    'final_train_acc': train_accs[-1],
    'final_test_acc': test_accs[-1],
    'final_train_loss': train_losses[-1],
    'final_test_loss': test_losses[-1],
    'roc_auc': roc_auc
}

res_path = f"results/{DATASET_NAME.lower()}_{MODEL_KEY}.json"
with open(res_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"Saved results to {res_path}")


## 11. Conclusion
Mô hình Basic CNN 1D đã được huấn luyện và đánh giá trên tập dữ liệu Diabetes (dữ liệu bảng coi như chuỗi 1D).
